# 185. Department Top Three Salaries

**Difficulty:** Hard &nbsp;|&nbsp; **Topics:** database, window-function, group-by
&nbsp;|&nbsp; [LeetCode](https://leetcode.com/problems/department-top-three-salaries/)

```
Table: Employee                        Table: Department
+--------------+---------+             +-------------+---------+
| Column Name  | Type    |             | Column Name | Type    |
+--------------+---------+             +-------------+---------+
| id           | int     |             | id          | int     |
| name         | varchar |             | name        | varchar |
| salary       | int     |             +-------------+---------+
| departmentId | int     |
+--------------+---------+
```

A company's executives are interested in seeing who earns the most money in each of the
company's departments. A **high earner** in a department is an employee who has a salary
in the **top three unique salaries** for that department.

Write a solution to find the employees who are high earners in each of the departments.

Return the result table in **any order**. The result columns must be called
`Department`, `Employee` and `Salary`.

---

### Example

```
Employee:                                   Department:
+----+-------+--------+--------------+      +----+-------+
| id | name  | salary | departmentId |      | id | name  |
+----+-------+--------+--------------+      +----+-------+
| 1  | Joe   | 85000  | 1            |      | 1  | IT    |
| 2  | Henry | 80000  | 2            |      | 2  | Sales |
| 3  | Sam   | 60000  | 2            |      +----+-------+
| 4  | Max   | 90000  | 1            |
| 5  | Janet | 69000  | 1            |
| 6  | Randy | 85000  | 1            |
| 7  | Will  | 70000  | 1            |
+----+-------+--------+--------------+

Output:
+------------+----------+--------+
| Department | Employee | Salary |
+------------+----------+--------+
| IT         | Max      | 90000  |
| IT         | Joe      | 85000  |
| IT         | Randy    | 85000  |
| IT         | Will     | 70000  |
| Sales      | Henry    | 80000  |
| Sales      | Sam      | 60000  |
+------------+----------+--------+
```

Look at IT carefully. The top three **unique** salaries are 90000, 85000, 70000. Joe and
Randy **both** earn 85000, so IT returns **four** employees for three salaries. Janet at
69000 misses out.

---

#184 with `3` instead of `1`, and the whole difficulty is that word **unique**. Get the
ranking function wrong and IT returns three people instead of four - and it will look
right.

## Before you write anything

**1.** For IT, write out the salaries in descending order: `90000, 85000, 85000, 70000,
69000`. Now number them three ways, by hand:

```
salary   ROW_NUMBER   RANK   DENSE_RANK
90000        1          1         1
85000        2          2         2
85000        3          2         2
70000        4          4         3
69000        5          5         4
```

Fill that table in yourself before reading on. Then answer: with a filter of `<= 3`,
which of the three columns produces the expected output, and exactly who does each of
the other two wrongly include or exclude?

**2.** Say in one sentence what each function does with a tie - `ROW_NUMBER` breaks it
arbitrarily, `RANK` shares the rank and then **skips**, `DENSE_RANK` shares the rank and
does **not** skip. Then map that onto the word **unique** in the statement. That mapping
is the entire problem.

**3.** `DENSE_RANK() OVER (PARTITION BY departmentId ORDER BY salary DESC)` - name what
each of the three pieces does. Which one makes the ranking restart for every department?
What would happen if you left it out?

**4.** Why can the ranking not go in the `WHERE` clause of the same `SELECT`? (The
evaluation order from #182: window functions are computed at `SELECT` time, after
`WHERE`.) So where does the filter have to go instead - name the two ways.

**5.** Departments with fewer than three distinct salaries: Sales has only two, and
**both** appear. Check that `<= 3` handles it with no special case, and say why.

**6.** Write the version with **no** window functions: for each employee, count how many
distinct salaries in their department are strictly greater than theirs, and keep the
ones where that count is less than 3. Convince yourself it is the same thing. That
correlated-subquery version is what people wrote before 2018, and being able to derive
it proves you understand what `DENSE_RANK` actually computes.

## Two routes

**A - `DENSE_RANK`** *(write this first)*

```sql
SELECT Department, Employee, Salary FROM (
    SELECT d.name AS Department, e.name AS Employee, e.salary AS Salary,
           DENSE_RANK() OVER (PARTITION BY e.departmentId ORDER BY e.salary DESC) AS rnk
    FROM Employee e
    JOIN Department d ON d.id = e.departmentId
) t
WHERE rnk <= 3
```

`PARTITION BY departmentId` restarts the ranking in every department. `ORDER BY salary
DESC` makes rank 1 the highest. `DENSE_RANK` gives equal salaries the same rank and does
**not** skip afterwards, which is exactly what "top three **unique** salaries" means -
so Joe and Randy share rank 2, Will gets rank 3, and IT correctly returns four people.

The subquery wrapper is question 4's answer: you cannot filter on a window function in
the same `WHERE`, because `WHERE` runs first.

**B - count the distinct salaries above you** *(no window functions)*

```sql
SELECT d.name AS Department, e.name AS Employee, e.salary AS Salary
FROM Employee e
JOIN Department d ON d.id = e.departmentId
WHERE (
    SELECT COUNT(DISTINCT e2.salary)
    FROM Employee e2
    WHERE e2.departmentId = e.departmentId AND e2.salary > e.salary
) < 3
```

"Fewer than three distinct salaries beat mine" is the same statement as "my dense rank
is at most 3", and writing both is the fastest way to be sure you know what
`DENSE_RANK` means. The `DISTINCT` inside `COUNT` is doing the same job the word *dense*
does - drop it and Joe and Randy start pushing each other down the list.

It runs a subquery per employee, so it is slower, and it is the answer you would have
had to give before window functions were widely available.

> **`ROW_NUMBER`, `RANK`, `DENSE_RANK` are three different answers to "what do ties
> mean?"** Nothing else separates them. Pick by reading the requirement: "top three
> rows" is `ROW_NUMBER`, "top three positions, skipping after ties" is `RANK`, "top
> three **distinct values**" is `DENSE_RANK`. This problem says unique salaries, so it
> is dense - and the test below with a four-way tie will tell you immediately if you
> reached for the wrong one.

In [ ]:
SOLUTION = '''
'''

### The test harness

Every notebook in this folder runs your SQL for real, against a fresh **SQLite**
database built from scratch for each test case. Nothing is mocked and nothing is
pattern-matched - if your query runs and returns the right rows, it passes.

`check(name, data, expected)` creates the tables, inserts that case's rows, executes
whatever string is in `SOLUTION`, and compares. It checks two things: the **rows**
(as a set - row order does not matter unless the problem says it does) and the
**column names**, because a query that returns the right numbers under the wrong
headings is not the answer the question asked for.

On failure it prints your rows next to the expected ones and names which rows are
missing and which should not be there.

`show(name, data, query)` is there for you: run *any* query against any dataset and
print it. Use it to look at intermediate results while you are working - especially
to run the deliberately-wrong version of your query and watch what it does.

> **SQLite here, MySQL on LeetCode.** They agree on everything these problems need -
> joins, `GROUP BY`/`HAVING`, subqueries, `LIMIT`/`OFFSET`, `COALESCE`, and window
> functions like `DENSE_RANK`. Where a problem needs something MySQL does differently,
> the notebook says so in the routes section. Write standard SQL and both will take it.

Run this cell; don't edit it.

In [ ]:
import sqlite3

SCHEMA = """CREATE TABLE Employee (id INTEGER, name TEXT, salary INTEGER, departmentId INTEGER);
CREATE TABLE Department (id INTEGER, name TEXT);"""

EXPECTED_COLUMNS = ['Department', 'Employee', 'Salary']
ORDERED = False


def _norm(rows):
    return rows if ORDERED else sorted(rows, key=lambda r: tuple((v is None, str(v)) for v in r))


def check(name, data_sql, expected):
    """Build a fresh in-memory database, run SOLUTION against it, compare."""
    con = sqlite3.connect(":memory:")
    try:
        con.executescript(SCHEMA)
        if data_sql.strip():
            con.executescript(data_sql)
    except sqlite3.Error as e:
        print(f"FAIL {name}")
        print(f"       the harness could not build the tables: {e}")
        return False

    if not SOLUTION.strip():
        print(f"FAIL {name}")
        print("       SOLUTION is empty - write your query in the cell above")
        return False

    try:
        cur = con.execute(SOLUTION)
        got = [tuple(r) for r in cur.fetchall()]
        cols = [d[0] for d in cur.description] if cur.description else []
    except sqlite3.Error as e:
        print(f"FAIL {name}")
        print(f"       your query raised {type(e).__name__}: {e}")
        return False

    cols_ok = [c.lower() for c in cols] == [c.lower() for c in EXPECTED_COLUMNS]
    rows_ok = _norm(got) == _norm(expected)

    if cols_ok and rows_ok:
        print(f"OK   {name}")
        return True

    print(f"FAIL {name}")
    if not cols_ok:
        print(f"       column names  {cols}")
        print(f"       should be     {EXPECTED_COLUMNS}")
    if not rows_ok:
        missing = [r for r in expected if r not in got]
        extra = [r for r in got if r not in expected]
        print(f"       you returned {len(got)} row(s), expected {len(expected)}"
              + ("   (row order matters here)" if ORDERED else "   (row order does not matter)"))
        for r in got[:6]:
            print(f"         got       {r}")
        for r in expected[:6]:
            print(f"         expected  {r}")
        if missing:
            print(f"       rows you are MISSING: {missing[:4]}")
        if extra:
            print(f"       rows you should NOT have: {extra[:4]}")
    return False


def show(name, data_sql, query):
    """Run any query against a dataset and print it - for exploring, not for grading."""
    con = sqlite3.connect(":memory:")
    con.executescript(SCHEMA)
    if data_sql.strip():
        con.executescript(data_sql)
    cur = con.execute(query)
    cols = [d[0] for d in cur.description]
    rows = cur.fetchall()
    print(f"-- {name}")
    print("   " + " | ".join(str(c) for c in cols))
    for r in rows:
        print("   " + " | ".join("NULL" if v is None else str(v) for v in r))
    if not rows:
        print("   (no rows)")

In [ ]:
# tests
check("the LeetCode example", '''
INSERT INTO Employee VALUES (1,'Joe',85000,1),(2,'Henry',80000,2),(3,'Sam',60000,2),
                            (4,'Max',90000,1),(5,'Janet',69000,1),(6,'Randy',85000,1),
                            (7,'Will',70000,1);
INSERT INTO Department VALUES (1,'IT'),(2,'Sales');
''', [('IT','Max',90000), ('IT','Joe',85000), ('IT','Randy',85000), ('IT','Will',70000),
      ('Sales','Henry',80000), ('Sales','Sam',60000)])

check("question 5: a department with fewer than three salaries", '''
INSERT INTO Employee VALUES (1,'A',10,1),(2,'B',20,1);
INSERT INTO Department VALUES (1,'Small');
''', [('Small','B',20), ('Small','A',10)])

check("exactly three distinct salaries", '''
INSERT INTO Employee VALUES (1,'A',30,1),(2,'B',20,1),(3,'C',10,1);
INSERT INTO Department VALUES (1,'Three');
''', [('Three','A',30), ('Three','B',20), ('Three','C',10)])

check("*** question 1: a four-way tie at the top - ROW_NUMBER fails here ***", '''
INSERT INTO Employee VALUES (1,'A',100,1),(2,'B',100,1),(3,'C',100,1),(4,'D',100,1),
                            (5,'E',50,1);
INSERT INTO Department VALUES (1,'Flat');
''', [('Flat','A',100), ('Flat','B',100), ('Flat','C',100), ('Flat','D',100),
      ('Flat','E',50)])

check("*** question 1: a tie in second place - RANK fails here ***", '''
INSERT INTO Employee VALUES (1,'A',100,1),(2,'B',90,1),(3,'C',90,1),(4,'D',80,1),
                            (5,'E',70,1);
INSERT INTO Department VALUES (1,'Tie');
''', [('Tie','A',100), ('Tie','B',90), ('Tie','C',90), ('Tie','D',80)])

check("question 3: ranking restarts per department", '''
INSERT INTO Employee VALUES (1,'A',10,1),(2,'B',9,1),(3,'C',8,1),(4,'D',7,1),
                            (5,'E',1000,2),(6,'F',999,2);
INSERT INTO Department VALUES (1,'Low'),(2,'High');
''', [('Low','A',10), ('Low','B',9), ('Low','C',8),
      ('High','E',1000), ('High','F',999)])

check("a department with no employees does not appear", '''
INSERT INTO Employee VALUES (1,'A',10,1);
INSERT INTO Department VALUES (1,'Busy'),(2,'Empty');
''', [('Busy','A',10)])

check("everybody in the department earns exactly the same", '''
INSERT INTO Employee VALUES (1,'A',5,1),(2,'B',5,1),(3,'C',5,1);
INSERT INTO Department VALUES (1,'Same');
''', [('Same','A',5), ('Same','B',5), ('Same','C',5)])

check("one employee only", '''
INSERT INTO Employee VALUES (1,'Solo',1,1);
INSERT INTO Department VALUES (1,'One');
''', [('One','Solo',1)])

check("no employees at all", '''
INSERT INTO Department VALUES (1,'Ghost');
''', [])

check("negative salaries rank the same way", '''
INSERT INTO Employee VALUES (1,'A',-1,1),(2,'B',-2,1),(3,'C',-3,1),(4,'D',-4,1);
INSERT INTO Department VALUES (1,'Debt');
''', [('Debt','A',-1), ('Debt','B',-2), ('Debt','C',-3)])

## After it passes

- **Run all three ranking functions side by side.** Take the "tie in second place"
  dataset and `show` a query selecting `salary`, `ROW_NUMBER()`, `RANK()` and
  `DENSE_RANK()` over the same window. Put the numbers next to the table you filled in
  by hand in question 1. Then swap each into your solution and watch which test fails -
  `ROW_NUMBER` on the four-way tie, `RANK` on the second-place tie. Those two datasets
  exist to tell the three functions apart, and after this you will never mix them up.
- **Write route B** and check it passes too. Then delete the `DISTINCT` from
  `COUNT(DISTINCT e2.salary)` and see which case breaks - that is the word *dense*, made
  visible.
- **Answer question 4 out loud.** Try putting `WHERE DENSE_RANK() OVER (...) <= 3`
  directly in the query and read the error. Then say where window functions sit in the
  evaluation order `FROM -> WHERE -> GROUP BY -> HAVING -> SELECT -> ORDER BY`.
- **Then generalise.** Make it top-N with a parameter. Add `ORDER BY Department, Salary
  DESC` to make it a readable report. Add `SUM(salary) OVER (PARTITION BY departmentId)`
  to show each earner's share of their department's payroll - one more window over the
  window you already wrote.
- Siblings: **#184 Department Highest Salary** (this at N=1 - do it first if you have
  not), #178 Rank Scores (`DENSE_RANK` with no partition), #177 Nth Highest Salary,
  #1077 Project Employees III, #602 Friend Requests II.